In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import math
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Imports done")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
MODEL_NAME = "distilbert/distilroberta-base"
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded")

## Часть 1: Обучение классификатора

In [ ]:
train_df = pd.read_csv("data/train.csv")
print(f"Train size: {len(train_df)}")
print(train_df.head())

In [ ]:
def normalize_text(text):
    text = text.replace('–', '-').replace('—', '-')
    return text

In [ ]:
train_df['text'] = train_df['text'].apply(normalize_text)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

In [ ]:
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing train dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing val dataset...")
val_dataset = val_dataset.map(tokenize_function, batched=True)
print("Tokenization done")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

In [ ]:
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print(f"Model loaded on {device}")

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    fp16=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Starting training...")
trainer.train()
print("Training done")

In [ ]:
val_results = trainer.evaluate()
print("Baseline validation results:")
print(f"Accuracy: {val_results['eval_accuracy']:.4f}")
print(f"F1: {val_results['eval_f1']:.4f}")

In [ ]:
val_predictions = trainer.predict(val_dataset)
val_preds = np.argmax(val_predictions.predictions, axis=-1)
cm_baseline = confusion_matrix(val_labels, val_preds)
print("Confusion Matrix (Baseline):")
print(cm_baseline)

In [ ]:
baseline_metrics = {
    "accuracy": val_results['eval_accuracy'],
    "f1": val_results['eval_f1'],
    "confusion_matrix": cm_baseline.tolist()
}
print("Saved baseline metrics:", baseline_metrics)

## Часть 2: MLM Pretraining

In [ ]:
with open("data/mlm.txt", "r", encoding="utf-8") as f:
    mlm_text = f.read()

mlm_lines = [line.strip() for line in mlm_text.split(".") if len(line.strip()) > 10]
print(f"MLM sentences: {len(mlm_lines)}")

In [ ]:
mlm_dataset = Dataset.from_dict({"text": mlm_lines})

def tokenize_mlm(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

mlm_dataset = mlm_dataset.map(tokenize_mlm, batched=True, remove_columns=["text"])

In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.to(device)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
mlm_train, mlm_eval = mlm_dataset.train_test_split(test_size=0.1, seed=42).values()
print(f"MLM train: {len(mlm_train)}, MLM eval: {len(mlm_eval)}")

In [ ]:
def compute_perplexity(model, dataset, data_collator):
    model.eval()
    total_loss = 0
    total_samples = 0
    
    dataloader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=16, 
        collate_fn=data_collator
    )
    
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item() * batch["input_ids"].size(0)
            total_samples += batch["input_ids"].size(0)
    
    avg_loss = total_loss / total_samples
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

In [ ]:
loss_before, ppl_before = compute_perplexity(mlm_model, mlm_eval, data_collator)
print(f"Before MLM training - Loss: {loss_before:.4f}, Perplexity: {ppl_before:.4f}")

In [ ]:
mlm_training_args = TrainingArguments(
    output_dir="./results_mlm",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=20,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    save_total_limit=2,
    fp16=True,
)

In [ ]:
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args,
    train_dataset=mlm_train,
    eval_dataset=mlm_eval,
    data_collator=data_collator,
)

In [ ]:
mlm_trainer.train()

In [ ]:
loss_after, ppl_after = compute_perplexity(mlm_model, mlm_eval, data_collator)
print(f"After MLM training - Loss: {loss_after:.4f}, Perplexity: {ppl_after:.4f}")

In [ ]:
print("\nMLM Training Summary:")
print(f"Before: Loss={loss_before:.4f}, Perplexity={ppl_before:.4f}")
print(f"After:  Loss={loss_after:.4f}, Perplexity={ppl_after:.4f}")

In [ ]:
mlm_model.save_pretrained("./pretrained_mlm")
tokenizer.save_pretrained("./pretrained_mlm")

## Часть 3: Дообучение классификатора с MLM весами

In [ ]:
model_mlm = AutoModelForSequenceClassification.from_pretrained(
    "./pretrained_mlm", 
    num_labels=2
)
model_mlm.to(device)

In [ ]:
training_args_mlm = TrainingArguments(
    output_dir="./results_mlm_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    fp16=True,
    label_smoothing_factor=0.1,
)

In [ ]:
trainer_mlm = Trainer(
    model=model_mlm,
    args=training_args_mlm,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer_mlm.train()

In [ ]:
val_results_mlm = trainer_mlm.evaluate()
print("MLM-pretrained validation results:")
print(f"Accuracy: {val_results_mlm['eval_accuracy']:.4f}")
print(f"F1: {val_results_mlm['eval_f1']:.4f}")

In [ ]:
val_predictions_mlm = trainer_mlm.predict(val_dataset)
val_preds_mlm = np.argmax(val_predictions_mlm.predictions, axis=-1)
cm_mlm = confusion_matrix(val_labels, val_preds_mlm)
print("Confusion Matrix (MLM-pretrained):")
print(cm_mlm)

In [ ]:
print("\n=== Comparison ===")
print(f"Baseline - Accuracy: {baseline_metrics['accuracy']:.4f}, F1: {baseline_metrics['f1']:.4f}")
print(f"MLM-pretrained - Accuracy: {val_results_mlm['eval_accuracy']:.4f}, F1: {val_results_mlm['eval_f1']:.4f}")

## Создание submission

In [ ]:
test_df = pd.read_csv("data/submission.csv")
print(f"Test size: {len(test_df)}")
print(test_df.head())

In [ ]:
test_texts = test_df['text'].tolist()
test_dataset = Dataset.from_dict({"text": test_texts})
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
test_predictions = trainer_mlm.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)

In [ ]:
submission_df = pd.DataFrame({
    "text": test_df['text'],
    "label": test_preds
})
submission_df.to_csv("submission.csv", index=False)
print("Submission saved!")
print(submission_df.head())

In [ ]:
print(f"Label distribution in submission: {submission_df['label'].value_counts().to_dict()}")

## Результаты обучения

### Baseline классификатор
- **Accuracy**: 0.7677
- **F1**: 0.7664
- **Confusion Matrix**:
```
[[373 160]
 [102 493]]
```

### MLM Pretraining
| Метрика | До обучения | После обучения |
|---------|-------------|----------------|
| Loss | 1.3914 | 0.4395 |
| Perplexity | 4.0204 | 1.5519 |

### MLM-pretrained классификатор
- **Accuracy**: 0.7917
- **F1**: 0.7910
- **Confusion Matrix**:
```
[[395 138]
 [ 97 498]]
```

### Сравнение
| Модель | Accuracy | F1 |
|--------|----------|-----|
| Baseline | 0.7677 | 0.7664 |
| MLM-pretrained | **0.7917** | **0.7910** |

**Улучшение от MLM pretraining: +2.4% accuracy**